<a href="https://colab.research.google.com/github/Dakshaayini/northstar-analytics/blob/main/Copy_of_01_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install packages (only need to do once per session)
install.packages(c('sqldf', 'DBI', 'RSQLite', 'dplyr', 'ggplot2', 'readr'))


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
# Cell 2: Load all libraries
library(sqldf)
library(DBI)
library(RSQLite)
library(dplyr)
library(ggplot2)
library(readr)

In [ ]:
# Cell 3: Load all NorthStar CSV files
customers  <- read.csv('customers.csv',  stringsAsFactors=FALSE)
orders     <- read.csv('orders.csv',     stringsAsFactors=FALSE)
deliveries <- read.csv('deliveries.csv', stringsAsFactors=FALSE)
drivers    <- read.csv('drivers.csv',    stringsAsFactors=FALSE)
vehicles   <- read.csv('vehicles.csv',   stringsAsFactors=FALSE)
hubs       <- read.csv('hubs.csv',       stringsAsFactors=FALSE)
incidents  <- read.csv('incidents.csv',  stringsAsFactors=FALSE)
complaints <- read.csv('complaints.csv', stringsAsFactors=FALSE)


In [ ]:
# Cell 4: Inspect the main delivery file
cat('Orders:', nrow(orders), '| Deliveries:', nrow(deliveries), '\n')
cat('Complaints:', nrow(complaints), '| Incidents:', nrow(incidents), '\n')

Orders: 1250 | Deliveries: 950 
Complaints: 320 | Incidents: 280 


In [ ]:
# Cell 5: SQL Query 1 — delivery failure rate by pickup zone
q1_zone <- sqldf("
  SELECT UPPER(o.pickup_zone)                           AS zone,
         COUNT(*)                                        AS total_deliveries,
         SUM(CASE WHEN d.delivery_status='Failed'
                  THEN 1 ELSE 0 END)                    AS failed_count,
         SUM(CASE WHEN d.delivery_status='Delayed'
                  THEN 1 ELSE 0 END)                    AS delayed_count,
         ROUND(100.0*SUM(CASE WHEN d.delivery_status='Failed'
               THEN 1 ELSE 0 END)/COUNT(*),1)           AS failure_rate_pct,
         ROUND(AVG(d.customer_rating_post_delivery),2)  AS avg_rating
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY UPPER(o.pickup_zone)
  ORDER BY failure_rate_pct DESC
")
print(q1_zone)

       zone total_deliveries failed_count delayed_count failure_rate_pct
1   CENTRAL              110           22            27             20.0
2       CTR               64           11            24             17.2
3     NORTH              135           22            21             16.3
4 RIVERSIDE              119           18            25             15.1
5      WEST              114           14            21             12.3
6      EAST              156           19            31             12.2
7   AIRPORT              113           12            31             10.6
8     SOUTH              139           14            22             10.1
  avg_rating
1       3.62
2       3.43
3       3.90
4       3.86
5       3.90
6       3.91
7       3.98
8       4.05


In [ ]:
# Cell 6: SQL Query 2 — manual override bands vs delivery outcome
q2_overrides <- sqldf("
  SELECT CASE
           WHEN manual_route_override_count=0 THEN '0 overrides'
           WHEN manual_route_override_count=1 THEN '1 override'
           WHEN manual_route_override_count BETWEEN 2 AND 3 THEN '2-3 overrides'
           ELSE '4+ overrides'
         END                                            AS override_band,
         COUNT(*)                                       AS total,
         SUM(CASE WHEN delivery_status='Failed' THEN 1 ELSE 0 END) AS failed,
         ROUND(100.0*SUM(CASE WHEN delivery_status='Failed'
               THEN 1 ELSE 0 END)/COUNT(*),1)          AS failure_rate_pct,
         ROUND(AVG(customer_rating_post_delivery),2)   AS avg_rating
  FROM deliveries
  GROUP BY override_band
  ORDER BY failure_rate_pct DESC
")
print(q2_overrides)

  override_band total failed failure_rate_pct avg_rating
1    1 override   310     51             16.5       3.88
2 2-3 overrides   210     32             15.2       3.78
3   0 overrides   399     46             11.5       3.90
4  4+ overrides    31      3              9.7       3.76


In [ ]:
# Cell 7: SQL Query 3 — customers with repeat complaints
q3_complaints <- sqldf("
  SELECT c.customer_id,
         cu.home_zone,
         cu.customer_type,
         COUNT(c.complaint_id)                                   AS total_complaints,
         SUM(CASE WHEN c.severity='High' THEN 1 ELSE 0 END)     AS high_severity,
         ROUND(SUM(c.compensation_amount),2)                     AS total_compensation
  FROM complaints c
  JOIN customers cu ON c.customer_id = cu.customer_id
  GROUP BY c.customer_id
  HAVING total_complaints >= 2
  ORDER BY total_complaints DESC, total_compensation DESC
  LIMIT 15
")
print(q3_complaints)

   customer_id home_zone customer_type total_complaints high_severity
1        C0368     North      Consumer                4             1
2        C0421   CENTRAL      Consumer                3             2
3        C0573   AIRPORT           SME                3             2
4        C0242      East      Consumer                3             1
5        C0282 RiverSide      Consumer                3             0
6        C0545     SOUTH      Consumer                3             0
7        C0372      West      Consumer                3             2
8        C0191     North      Consumer                3             0
9        C0172     north      Consumer                3             1
10       C0142     SOUTH      Consumer                3             1
11       C0110      EAST      Consumer                3             1
12       C0626     SOUTH      Consumer                3             1
13       C0351   CENTRAL    Enterprise                2             2
14       C0078      

In [ ]:
con <- dbConnect(RSQLite::SQLite(), ':memory:')
dbWriteTable(con, 'deliveries', deliveries)
dbWriteTable(con, 'orders',     orders)

# Check query plan BEFORE index (should say SCAN TABLE)
before_idx <- dbGetQuery(con,
  'EXPLAIN QUERY PLAN
   SELECT * FROM deliveries WHERE delivery_status = \'Failed\'')
print('BEFORE INDEX:'); print(before_idx)

# Create indexes and check after index, with error handling
tryCatch({
  dbExecute(con, 'CREATE INDEX idx_status   ON deliveries (delivery_status)')
  dbExecute(con, 'CREATE INDEX idx_override  ON deliveries (manual_route_override_count)')
  dbExecute(con, 'CREATE INDEX idx_hub       ON deliveries (hub_id)')
  dbExecute(con, 'CREATE INDEX idx_order_del ON deliveries (order_id)')

  # Check query plan AFTER index (should say USING INDEX)
  after_idx <- dbGetQuery(con,
    'EXPLAIN QUERY PLAN
     SELECT * FROM deliveries WHERE delivery_status = \'Failed\'')
  print('AFTER INDEX:'); print(after_idx)

  # List all indexes
  index_list <- dbGetQuery(con, 'PRAGMA index_list(deliveries)')
  print('INDEX LIST:'); print(index_list)

}, error = function(e) {
  message("An error occurred during index creation or post-index query:")
  message(e$message)
})

dbDisconnect(con)

[1] "BEFORE INDEX:"
  id parent notused          detail
1  2      0     216 SCAN deliveries


[1] 0

[1] 0

[1] 0

[1] 0

[1] "AFTER INDEX:"
  id parent notused
1  3      0      61
                                                        detail
1 SEARCH deliveries USING INDEX idx_status (delivery_status=?)


seq,name,unique,origin,partial
<int>,<chr>,<int>,<chr>,<int>
0,idx_order_del,0,c,0
1,idx_hub,0,c,0
2,idx_override,0,c,0
3,idx_status,0,c,0
